# RQ1 Multi-GAN Model Comparison

This notebook launches or resumes the CTAB-GAN+, CTGAN, and non-private DP-CGANS-architecture baseline comparison and displays its aggregate artifacts. For a long GPU run, launch the same printed command through `nohup` or SLURM so it survives a browser or VPN disconnect.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys
import pandas as pd
from IPython.display import Image, display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
CONFIG = PROJECT_ROOT / 'configs' / 'rq1_mimic.json'
OUTPUT_DIR = PROJECT_ROOT / 'results' / 'rq1_mimic_notebook'
DEVICE = 'auto'
STAGE = 'val'
MODELS = 'ctabgan_plus,ctgan,dp_cgan'
SEEDS = '42,43,44'
SMOKE = True
RESUME = False
RUN_EXPERIMENT = False

In [ ]:
command = [
    sys.executable, '-u', '-m', 'xai_reweighting.run_model_comparison',
    '--config', str(CONFIG), '--stage', STAGE, '--device', DEVICE,
    '--models', MODELS, '--seeds', SEEDS, '--output-dir', str(OUTPUT_DIR),
    '--progress', 'auto',
]
if SMOKE:
    command.append('--smoke')
if RESUME:
    command.append('--resume')
print(' '.join(command))
if RUN_EXPERIMENT:
    subprocess.run(command, cwd=PROJECT_ROOT, check=True)

## Aggregate results

In [ ]:
summary_path = OUTPUT_DIR / 'rq1_summary.csv'
results_path = OUTPUT_DIR / 'rq1_results.csv'
if summary_path.exists():
    summary = pd.read_csv(summary_path)
    display(summary)
    display(pd.read_csv(results_path))
else:
    print('No results yet. Set RUN_EXPERIMENT=True or point OUTPUT_DIR to a completed run.')

## Fidelity, tails, utility, and privacy-proxy plots

In [ ]:
for name in ('utility', 'fidelity', 'tails', 'privacy_proxy'):
    path = OUTPUT_DIR / 'plots' / f'rq1_{name}.png'
    if path.exists():
        display(Image(filename=str(path)))

## Non-private DP-CGANS baseline settings and training diagnostics

In [ ]:
for seed in [int(value) for value in SEEDS.split(',')]:
    run_dir = OUTPUT_DIR / 'models' / 'dp_cgan' / f'seed_{seed}'
    model_config = run_dir / 'model_config.json'
    history = run_dir / 'training_history.csv'
    if model_config.exists():
        print(f'Non-private DP-CGANS baseline seed {seed}')
        display(pd.Series(json.loads(model_config.read_text())))
    if history.exists():
        frame = pd.read_csv(history)
        if not frame.empty:
            display(frame.tail())

The DP-CGANS package is run with `private=false`. No epsilon, delta, noise multiplier, clipping, or formal privacy guarantee applies; privacy-proxy evaluation remains part of the common model comparison.